# 🌸 Marigold 360° Panorama Depth Estimation & 3D Reconstruction

GPU-accelerated, production-ready inference notebook for **[marigold_cli](https://github.com/A511-git/marigold_cli)**.

### 🌟 Pipeline Overview
- **Package Manager**: High-speed environment synchronization via `uv sync` from `pyproject.toml`.
- **12-Camera Icosahedral Spherical Splitter**: Decomposes 360° equirectangular panoramas into perspective views without polar distortion.
- **Marigold Diffusion Monocular Depth Estimation**: High-precision depth prediction via Latent Diffusion models.
- **Multi-Scale Poisson Gradient Solver**: Solves overdetermined spherical gradient equations for seamless, artifact-free 360° panoramic depth.
- **Export Options**: `depth.npy` (Float32), `depth.exr` (HDR), `depth_vis.png` (Colorized Turbo), `mask.png`, and `custom_d2p.ply` (3D Point Cloud).
- **Multi-GPU Parallel Queue**: Automatically distributes batch processing across dual GPUs (e.g. 2x T4 on Kaggle).
- **KaggleHub Integration**: Direct one-click dataset export & upload to Kaggle Datasets.

In [ ]:
# =============================================================================
# 1. CONFIGURATION & PARAMETERS
# =============================================================================
import os
import torch

# ---- Input & Output Paths ----
INPUT_DIR   = "/kaggle/input/datasets/newmailserver/panorama-imgs/panno"   # dataset attached to notebook
WORK_DIR    = "/kaggle/working/marigold_cli"                              # where marigold_cli repo gets cloned
OUTPUT_DIR  = "/kaggle/working/output"                                    # root output folder

# ---- Git Repository & Model Checkpoint ----
MARIGOLD_REPO       = "https://github.com/A511-git/marigold_cli"
MARIGOLD_CHECKPOINT = "prs-eth/marigold-depth-v1-1"  # Options: "prs-eth/marigold-depth-v1-1" (Quality) or "prs-eth/marigold-depth-lcm-v1-0" (Fast LCM)

# ---- Inference Settings ----
USE_DIFFUSERS       = True     # Use Hugging Face Diffusers Marigold pipeline
USE_FP16            = True     # Use FP16 half precision for faster inference & lower VRAM
SPLIT_RESOLUTION    = 512      # Resolution per perspective tile (512 for speed, 1024 for max quality)
RESIZE_RESOLUTION   = None     # Max dimension ceiling for input panorama (or None to keep original)
BATCH_SIZE          = 1        # Batch size for perspective view inference
SAVE_MAPS           = True     # Save depth_vis.png, depth.exr, and mask.png
SAVE_POINTS_PLY     = True     # Save 3D point cloud pointcloud.ply
SAVE_DEBUG          = False    # Save 12 individual splitted perspective views and camera JSONs

# ---- GPU Assignment ----
# Kaggle gives 2x T4 (cuda:0, cuda:1), or auto-detect available GPUs
num_gpus = torch.cuda.device_count()
GPU_IDS = list(range(num_gpus)) if num_gpus > 0 else [0]

# Supported image extensions
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP")

# ---- Kaggle Dataset Upload Configuration (Optional) ----
KAGGLE_USERNAME = "newmailserver"
DATASET_SLUG    = "marigold-pano-output"
UPLOAD_DIR      = "/kaggle/temp/upload"   # staging dir containing output + metadata

# Create base directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Config loaded successfully.")
print("Input dir: ", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Repo:      ", MARIGOLD_REPO)
print("Model:     ", MARIGOLD_CHECKPOINT)
print("GPU IDs:   ", GPU_IDS)

In [ ]:
# =============================================================================
# 2. INSTALLATION (UV SYNC FROM PYPROJECT.TOML)
# =============================================================================
import os

# Sanitize backend environment variable
os.environ["MPLBACKEND"] = "Agg"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

!pip install -q uv

%cd {WORK_DIR}/..
!rm -rf {WORK_DIR}
!git clone --depth 1 {MARIGOLD_REPO} {WORK_DIR}

%cd {WORK_DIR}
!uv sync

# Verify installation via uv virtual environment
!uv run python -c "import torch, diffusers, cv2, standalone_marigold; print('All Marigold modules and dependencies imported successfully via uv!')"

# Post-processing & KaggleHub dependencies in the base environment
!pip install -q opencv-python Pillow numpy imageio kagglehub

print("\nInstall complete.")

In [ ]:
# =============================================================================
# 3. MULTI-GPU INFERENCE MANAGER
# =============================================================================
import glob
import os
import queue
import subprocess
import threading
import time

# Collect all images from the dataset
image_paths = sorted([
    p for p in glob.glob(os.path.join(INPUT_DIR, "**", "*"), recursive=True)
    if p.lower().endswith(IMAGE_EXTS)
])
print(f"Found {len(image_paths)} images.")

work_q = queue.Queue()
for p in image_paths:
    work_q.put(p)

results_lock = threading.Lock()
completed = []
failed = []

def get_out_parent(img_path):
    """Parent dir to hand to CLI; creates the <img_name> folder inside it."""
    rel_path = os.path.relpath(img_path, INPUT_DIR)
    rel_dir, _ = os.path.split(rel_path)
    return os.path.join(OUTPUT_DIR, rel_dir) if rel_dir else OUTPUT_DIR
    
def get_out_root(img_path):
    """Final result folder after CLI runs: OUTPUT_DIR/<rel_dir>/<img_name>."""
    rel_path = os.path.relpath(img_path, INPUT_DIR)
    rel_dir, fname = os.path.split(rel_path)
    img_name = os.path.splitext(fname)[0]
    return os.path.join(OUTPUT_DIR, rel_dir, img_name) if rel_dir else os.path.join(OUTPUT_DIR, img_name)

def run_marigold_on_gpu(device_id):
    while True:
        try:
            img_path = work_q.get_nowait()
        except queue.Empty:
            return

        out_parent = get_out_parent(img_path)
        os.makedirs(out_parent, exist_ok=True)
        img_name = os.path.relpath(get_out_root(img_path), OUTPUT_DIR)

        # Execute Marigold-360 CLI via uv run in WORK_DIR
        cmd = [
            "uv", "run", "python", "app.py",
            "-i", img_path,
            "-o", out_parent,
            "-c", MARIGOLD_CHECKPOINT,
            "--device", f"cuda:{device_id}",
            "--split_resolution", str(SPLIT_RESOLUTION),
            "--batch_size", str(BATCH_SIZE),
        ]

        if USE_DIFFUSERS:
            cmd.append("--diffusers")
        if USE_FP16:
            cmd.append("--fp16")
        if RESIZE_RESOLUTION is not None:
            cmd.extend(["--resize", str(RESIZE_RESOLUTION)])
        if SAVE_MAPS:
            cmd.append("--maps")
        if SAVE_POINTS_PLY:
            cmd.append("--points_ply")
        if SAVE_DEBUG:
            cmd.append("--debug")

        env = os.environ.copy()
        env["MPLBACKEND"] = "Agg"
        env["OPENCV_IO_ENABLE_OPENEXR"] = "1"

        t_start = time.time()
        print(f"[GPU {device_id}] 🚀 Starting: {img_name}")
        result = subprocess.run(
            cmd,
            cwd=WORK_DIR,
            env=env,
            capture_output=True,
            text=True
        )
        elapsed = time.time() - t_start

        with results_lock:
            if result.returncode == 0:
                completed.append(img_name)
                print(f"[GPU {device_id}] ✅ Done in {elapsed:.1f}s: {img_name}")
            else:
                failed.append((img_name, result.stderr[-500:]))
                print(f"[GPU {device_id}] ❌ Failed: {img_name}")
                print(result.stderr[-500:])

        work_q.task_done()

if image_paths:
    start = time.time()
    threads = [threading.Thread(target=run_marigold_on_gpu, args=(gid,)) for gid in GPU_IDS]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    print(f"\nDone in {time.time()-start:.1f}s")
    print(f"Completed: {len(completed)}  Failed: {len(failed)}")
    if failed:
        print("Failed images:", [f[0] for f in failed])

In [ ]:
# =============================================================================
# 4. D2P POST-PROCESSING (3D POINT CLOUDS & SURFACE NORMALS)
# =============================================================================
import os
import glob
from typing import Optional, Tuple, Union
import numpy as np
from PIL import Image
import cv2

def image_uv(
    width: int,
    height: int,
    left: Optional[int] = None,
    top: Optional[int] = None,
    right: Optional[int] = None,
    bottom: Optional[int] = None,
    dtype: np.dtype = np.float32
) -> np.ndarray:
    if left is None: left = 0
    if top is None: top = 0
    if right is None: right = width
    if bottom is None: bottom = height
    u = np.linspace((left + 0.5) / width, (right - 0.5) / width, right - left, dtype=dtype)
    v = np.linspace((top + 0.5) / height, (bottom - 0.5) / height, bottom - top, dtype=dtype)
    u, v = np.meshgrid(u, v, indexing='xy')
    return np.stack([u, v], axis=2)

def sphere_uv2dirs(uv: np.ndarray) -> np.ndarray:
    theta = (1.0 - uv[..., 0]) * (2.0 * np.pi)
    phi = uv[..., 1] * np.pi
    directions = np.stack([
        np.sin(phi) * np.cos(theta),
        np.sin(phi) * np.sin(theta),
        np.cos(phi)
    ], axis=-1)
    return directions

def points_to_normals(point: np.ndarray, mask: Optional[np.ndarray] = None) -> Union[np.ndarray, Tuple[np.ndarray, np.ndarray]]:
    height, width = point.shape[-3:-1]
    has_mask = mask is not None

    if mask is None:
        mask = np.ones((height, width), dtype=bool)
    else:
        mask = mask.astype(bool)

    mask_pad = np.zeros((height + 2, width + 2), dtype=bool)
    mask_pad[1:-1, 1:-1] = mask

    pts = np.zeros((height + 2, width + 2, 3), dtype=point.dtype)
    pts[1:-1, 1:-1, :] = point

    up = pts[:-2, 1:-1, :] - pts[1:-1, 1:-1, :]
    left = pts[1:-1, :-2, :] - pts[1:-1, 1:-1, :]
    down = pts[2:, 1:-1, :] - pts[1:-1, 1:-1, :]
    right = pts[1:-1, 2:, :] - pts[1:-1, 1:-1, :]

    normal = np.stack([
        np.cross(up, left, axis=-1),
        np.cross(left, down, axis=-1),
        np.cross(down, right, axis=-1),
        np.cross(right, up, axis=-1),
    ])
    normal = normal / (np.linalg.norm(normal, axis=-1, keepdims=True) + 1e-12)

    valid = np.stack([
        mask_pad[:-2, 1:-1] & mask_pad[1:-1, :-2],
        mask_pad[1:-1, :-2] & mask_pad[2:, 1:-1],
        mask_pad[2:, 1:-1] & mask_pad[1:-1, 2:],
        mask_pad[1:-1, 2:] & mask_pad[:-2, 1:-1],
    ]) & mask_pad[None, 1:-1, 1:-1]

    normal = (normal * valid[..., None]).sum(axis=0)
    normal = normal / (np.linalg.norm(normal, axis=-1, keepdims=True) + 1e-12)

    if has_mask:
        normal_mask = valid.any(axis=0)
        normal = np.where(normal_mask[..., None], normal, 0)
        return normal, normal_mask
    else:
        return normal

def normal_normalize(normal: np.ndarray) -> np.ndarray:
    normal_norm = np.linalg.norm(normal, axis=-1, keepdims=True)
    normal_norm[normal_norm < 1e-6] = 1e-6
    return normal / normal_norm

def colorize_normal(normal: np.ndarray, normal_mask: np.ndarray) -> np.ndarray:
    normal_rgb = (((normal + 1.0) * 0.5) * 255.0).astype(np.uint8)
    normal_mask_3d = np.repeat(np.expand_dims(normal_mask, axis=-1), 3, axis=-1).astype(np.uint8)
    return normal_rgb * normal_mask_3d

def save_3d_points_ply(points: np.ndarray, colors: np.ndarray, mask: np.ndarray, save_path: str, binary: bool = True):
    """Saves fast binary or ascii PLY."""
    points = points.reshape(-1, 3)
    colors = colors.reshape(-1, 3)
    flat_mask = mask.reshape(-1).astype(bool)

    valid_points = points[flat_mask]
    valid_colors = colors[flat_mask]

    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)

    num_vertices = len(valid_points)
    if binary:
        header = (
            f"ply\n"
            f"format binary_little_endian 1.0\n"
            f"comment vertices with color\n"
            f"element vertex {num_vertices}\n"
            f"property float x\n"
            f"property float y\n"
            f"property float z\n"
            f"property uchar red\n"
            f"property uchar green\n"
            f"property uchar blue\n"
            f"end_header\n"
        ).encode('ascii')
        vertex_dtype = np.dtype([
            ('x', '<f4'), ('y', '<f4'), ('z', '<f4'),
            ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')
        ])
        data = np.empty(num_vertices, dtype=vertex_dtype)
        data['x'] = valid_points[:, 0]
        data['y'] = valid_points[:, 1]
        data['z'] = valid_points[:, 2]
        data['red'] = valid_colors[:, 0].astype(np.uint8)
        data['green'] = valid_colors[:, 1].astype(np.uint8)
        data['blue'] = valid_colors[:, 2].astype(np.uint8)
        with open(save_path, 'wb') as f:
            f.write(header)
            f.write(data.tobytes())
    else:
        header = (
            f"ply\n"
            f"format ascii 1.0\n"
            f"comment vertices with color\n"
            f"element vertex {num_vertices}\n"
            f"property float x\n"
            f"property float y\n"
            f"property float z\n"
            f"property uchar red\n"
            f"property uchar green\n"
            f"property uchar blue\n"
            f"end_header\n"
        )
        with open(save_path, 'w', encoding='ascii') as f:
            f.write(header)
            for pt, cl in zip(valid_points, valid_colors):
                f.write(f"{pt[0]:.6f} {pt[1]:.6f} {pt[2]:.6f} {int(cl[0])} {int(cl[1])} {int(cl[2])}\n")

def distance2pointcloud(
    distance: np.ndarray,
    image: np.ndarray,
    mask: Optional[np.ndarray] = None,
    save_path: Optional[str] = None,
    return_normal: bool = False,
    save_distance: bool = False,
    binary_ply: bool = True
):
    if distance.ndim >= 3:
        distance = distance.squeeze()

    if mask is None:
        mask = (distance > 0) & np.isfinite(distance)

    if save_distance and save_path is not None:
        save_path_dis = save_path.replace('3dpc', 'depth').replace('.ply', '.npy')
        save_dir_dis = os.path.dirname(save_path_dis)
        if save_dir_dis and not os.path.exists(save_dir_dis):
            os.makedirs(save_dir_dis, exist_ok=True)
        np.save(save_path_dis, distance)

    height, width = distance.shape[:2]
    uv = image_uv(width=width, height=height)
    points = distance[:, :, None] * sphere_uv2dirs(uv)

    if save_path is not None:
        save_3d_points_ply(points, image, mask, save_path, binary=binary_ply)

    if return_normal:
        normal, normal_mask = points_to_normals(points, mask)
        normal = normal * np.array([-1, -1, 1])
        normal = normal_normalize(normal)
        normal_1 = normal[..., 0]
        normal_2 = normal[..., 1]
        normal_3 = normal[..., 2]
        normal = np.stack([normal_1, normal_3, normal_2], axis=-1)
        normal_img = colorize_normal(normal, normal_mask)
        return Image.fromarray(normal_img)

    return points

def find_depth_file(img_output_root):
    """Finds merged depth.npy or depth.exr, excluding splitted/ crops."""
    npy_path = os.path.join(img_output_root, "depth.npy")
    if os.path.exists(npy_path):
        return npy_path
    candidates = glob.glob(os.path.join(img_output_root, "**", "depth.exr"), recursive=True)
    candidates = [c for c in candidates if f"{os.sep}splitted{os.sep}" not in c]
    return candidates[0] if candidates else None

def process_folder(img_output_root, original_image_path, return_normal=False, binary_ply=True):
    depth_path = find_depth_file(img_output_root)
    if depth_path is None:
        return False, "no depth.npy or depth.exr found"

    if depth_path.endswith('.npy'):
        depth = np.load(depth_path)
    else:
        depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        if depth is None:
            try:
                import imageio.v3 as iio
                depth = iio.imread(depth_path)
            except Exception:
                pass
        if depth is not None and depth.ndim == 3:
            depth = depth[:, :, 0]

    if depth is None:
        return False, f"could not read {depth_path}"

    height, width = depth.shape[:2]
    rgb_pil = Image.open(original_image_path).convert("RGB")
    if rgb_pil.size != (width, height):
        rgb_pil = rgb_pil.resize((width, height), Image.Resampling.BILINEAR)
    image_rgb = np.array(rgb_pil)

    out_ply = os.path.join(img_output_root, "custom_d2p.ply")
    distance2pointcloud(
        distance=depth,
        image=image_rgb,
        mask=None,
        save_path=out_ply,
        return_normal=return_normal,
        save_distance=False,
        binary_ply=binary_ply
    )
    return True, out_ply

# Run over every image's output folder
d2p_results = []
for img_path in image_paths:
    img_output_root = get_out_root(img_path)
    if not os.path.isdir(img_output_root):
        continue

    rel_name = os.path.relpath(img_output_root, OUTPUT_DIR)
    ok, info = process_folder(img_output_root, img_path, return_normal=False, binary_ply=True)
    d2p_results.append((rel_name, ok, info))
    status = "✅" if ok else "❌"
    print(f"{status} {rel_name}: {info}")

n_ok = sum(1 for _, ok, _ in d2p_results if ok)
print(f"\nD2P complete: {n_ok}/{len(d2p_results)} succeeded.")

In [ ]:
# =============================================================================
# 5. UPLOAD FOLDER DIRECTLY TO KAGGLEHUB (NO ZIP)
# =============================================================================
import shutil
import json
import kagglehub

# 1. Prepare Staging Upload Directory
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)

shutil.copytree(OUTPUT_DIR, UPLOAD_DIR)
n_copied = sum(1 for _, dirs, files in os.walk(UPLOAD_DIR) if files)
print(f"Copied {n_copied} output folders into {UPLOAD_DIR}")

# 2. Write Metadata
handle = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
metadata = {
    "title": "Marigold 360 Panorama Output",
    "id": handle,
    "licenses": [{"name": "CC0-1.0"}],
    "subtitle": "Depth maps, EXRs, and custom D2P PLYs from Marigold 360 panorama inference",
    "description": (
        f"Output of Marigold 360 ({MARIGOLD_CHECKPOINT}) panorama inference run on "
        f"{len(image_paths)} panorama images across dual T4 GPUs, plus custom-generated "
        f"D2P point clouds (custom_d2p.ply) derived from each image's depth map."
    ),
    "keywords": ["marigold", "depth-estimation", "point-cloud", "panorama", "3d"]
}

with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# 3. Upload Directly to Kaggle Datasets
try:
    kagglehub.dataset_upload(handle, UPLOAD_DIR, version_notes="Marigold 360 + D2P batch output")
    print(f"✅ SUCCESS: https://www.kaggle.com/datasets/{handle}")
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}: {e}")